In [14]:
import pandas as pd
import numpy as np

data = {
    'Transaction_ID': range(1, 11),
    'Product_Category': ['Electronics', 'Home', 'Electronics', 'Sports', 'Home', 
                         'Electronics', 'Home', 'Sports', 'Electronics', 'Electronics'],
    'Sales_Amount': [150, 200, 155, 300, 210, 180, 205, 1000, 190, 160],  # 1000 is an Outlier
    'Customer_Age': [25, 34, np.nan, 45, 23, 31, 29, np.nan, 38, 40],     # Contains Nulls
    'Rating': [5, 4, 3, 5, 2, 4, 5, 2, 4, 3]
}

df_test = pd.DataFrame(data)

def automated_stat_analyzer(df, column_name):
    """
    Summarize central tendency, dispersion, and skewness for numerical columns, or mode for categorical.
    """
    col = df[column_name]
    
    if pd.api.types.is_numeric_dtype(col):
        mean_val = col.mean()
        median_val = col.median()
        std_val = col.std()
        
        if mean_val > median_val:
            skewness = "Right-Skewed"
        elif mean_val < median_val:
            skewness = "Left-Skewed"
        else:
            skewness = "Symmetric"
            
        return {
            "Type": "Numerical",
            "Mean": mean_val,
            "Median": median_val,
            "Standard Deviation": std_val,
            "Skewness": skewness
        }
    else:
        mode_val = col.mode()[0] if not col.mode().empty else None
        return {
            "Type": "Categorical",
            "Mode": mode_val
        }

sales_report = automated_stat_analyzer(df_test, 'Sales_Amount')
print("--- Numerical Test (Sales_Amount) ---")
print(sales_report)

cat_report = automated_stat_analyzer(df_test, 'Product_Category')
print("\n--- Categorical Test (Product_Category) ---")
print(cat_report)

--- Numerical Test (Sales_Amount) ---
{'Type': 'Numerical', 'Mean': np.float64(275.0), 'Median': np.float64(195.0), 'Standard Deviation': np.float64(258.30645021412494), 'Skewness': 'Right-Skewed'}

--- Categorical Test (Product_Category) ---
{'Type': 'Categorical', 'Mode': 'Electronics'}


In [23]:

def null_handling_strategy(df, strategy="fill_mean"):
    """
    Handle missing values by dropping rows or imputing numeric columns with mean/median.
    """
    df_cleaned = df.copy()
    
    if strategy == "drop_rows":
        return df_cleaned.dropna()
        
    elif strategy == "fill_mean":
        num_cols = df_cleaned.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mean())
        return df_cleaned
        
    elif strategy == "fill_median":
        num_cols = df_cleaned.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
        return df_cleaned
        
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

df_mean = null_handling_strategy(df_test, strategy="fill_mean")
print("--- Fill Mean Result (Customer_Age) ---")
print(df_mean[['Transaction_ID', 'Customer_Age']])

df_median = null_handling_strategy(df_test, strategy="fill_median")
print("\n--- Fill Median Result (Customer_Age) ---")
print(df_median[['Transaction_ID', 'Customer_Age']])

df_dropped = null_handling_strategy(df_test, strategy="drop_rows")
print("\n--- Drop Rows Result (Shape changed from 10 to 8 rows) ---")
print(f"Original rows: {len(df_test)}, New rows: {len(df_dropped)}")

--- Fill Mean Result (Customer_Age) ---
   Transaction_ID  Customer_Age
0               1        25.000
1               2        34.000
2               3        33.125
3               4        45.000
4               5        23.000
5               6        31.000
6               7        29.000
7               8        33.125
8               9        38.000
9              10        40.000

--- Fill Median Result (Customer_Age) ---
   Transaction_ID  Customer_Age
0               1          25.0
1               2          34.0
2               3          32.5
3               4          45.0
4               5          23.0
5               6          31.0
6               7          29.0
7               8          32.5
8               9          38.0
9              10          40.0

--- Drop Rows Result (Shape changed from 10 to 8 rows) ---
Original rows: 10, New rows: 8
